## Imports

In [1]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## Connection

In [2]:
presto_port = '80'
# username = 'airflow-user'
username = 'manoj.ravirajan@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3 = presto.connect('processing-2.processing.data.production.internal', presto_port, username)
connection4 = presto.connect('processing.processing.data.production.internal', presto_port, username)

## Parameter

In [3]:
city = 'Bangalore'
date_list = ['2024-09-20']
base_week = date_list[0]

## Dataset & User-Defined Functions

In [4]:
def consideration_query(run_date, city_name, base_week):
    
    rr_data_query = f"""
  
        with dormant_customer as (
            
            select
                customer_id as customer,
                date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) rapido_first_ride,
                ceil(cast(date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) as real)/10)*10 rapido_age_seg_fr,
                
                taxi_lifetime_last_ride_date, taxi_lifetime_first_ride_date, 
                taxi_lifetime_stage, previous_taxi_lifetime_stage, taxi_income_segment,
                taxi_lifetime_rides, taxi_recency_segment, taxi_rr_active_days_last_21_days as run_taxi_rr_active_days_last_21_days,
                taxi_lifetime_last_ride_city city_name, 
                
                fe_mean_intent_type, fe_intent_trend_type, fe_potential_per_type, 
                fe_regularity_type, fe_recency_type, fe_volatility_type,
                
                gender_tag, ps_tag_link, ps_tag_auto,
                
                taxi_service_affinity, taxi_distance_preference, taxi_time_of_day_preference, taxi_day_of_week_preference
            
            from 
                datasets.iallocator_customer_segments
            where 
                date(run_date) = date('{base_week}')
                -- and taxi_lifetime_last_ride_date = date('2023-11-01')
                and taxi_recency_segment in ('RECENT')
                and taxi_lifetime_rides > 4
                and taxi_lifetime_last_ride_city in ('{city_name}')

                and customer_id  = '6288c6694281e53bbe33d133'
                -- IN ('63058bc3ca3c325e3991febd', '650c1afb1213a91bc9edf1a0', '665c1b793983e81f8c129b31', '63db6fe7beabc2a4ef03caa4', '654c65e15f58b4d23f3d6f8d', '66da8a60a1d537cc59aed940', '63a2121398760530cc3fe0f2', '63f8e3da959e097a43c4befb')
        ),
        
        base as (
        
            select 
                a.*,
                case when coalesce(b.rr,0) <1 then 0 else 1 end as rr_retention_flag,
                case when coalesce(b.fe,0) <1 then 0 else 1 end as fe_retention_flag,
                case when coalesce(b.ao,0) <1 then 0 else 1 end as ao_retention_flag,
                case when coalesce(b.nr,0) <1 then 0 else 1 end as net_retention_flag,
                
                case when coalesce(b.nr_post_w01,0) <1 then 0 else 1 end as nr_flag_w01,
                case when coalesce(b.nr_post_w02,0) <1 then 0 else 1 end as nr_flag_w02,
                case when coalesce(b.nr_post_w03,0) <1 then 0 else 1 end as nr_flag_w03,
                case when coalesce(b.nr_post_w04,0) <1 then 0 else 1 end as nr_flag_w04,
                case when coalesce(b.nr_post_w05,0) <1 then 0 else 1 end as nr_flag_w05,
                case when coalesce(b.nr_post_w06,0) <1 then 0 else 1 end as nr_flag_w06,
                case when coalesce(b.nr_post_w07,0) <1 then 0 else 1 end as nr_flag_w07,
                case when coalesce(b.nr_post_w08,0) <1 then 0 else 1 end as nr_flag_w08,
                case when coalesce(b.nr_post_w09,0) <1 then 0 else 1 end as nr_flag_w09,
                case when coalesce(b.nr_post_w10,0) <1 then 0 else 1 end as nr_flag_w10,
                
                case when coalesce(b.rr_post_w01,0) <1 then 0 else 1 end as rr_flag_w01,
                case when coalesce(b.rr_post_w02,0) <1 then 0 else 1 end as rr_flag_w02,
                case when coalesce(b.rr_post_w03,0) <1 then 0 else 1 end as rr_flag_w03,
                case when coalesce(b.rr_post_w04,0) <1 then 0 else 1 end as rr_flag_w04,
                case when coalesce(b.rr_post_w05,0) <1 then 0 else 1 end as rr_flag_w05,
                case when coalesce(b.rr_post_w06,0) <1 then 0 else 1 end as rr_flag_w06,
                case when coalesce(b.rr_post_w07,0) <1 then 0 else 1 end as rr_flag_w07,
                case when coalesce(b.rr_post_w08,0) <1 then 0 else 1 end as rr_flag_w08,
                case when coalesce(b.rr_post_w09,0) <1 then 0 else 1 end as rr_flag_w09,
                case when coalesce(b.rr_post_w10,0) <1 then 0 else 1 end as rr_flag_w10,
                
                case when coalesce(b.fe_post_w01,0) <1 then 0 else 1 end as fe_flag_w01,
                case when coalesce(b.fe_post_w02,0) <1 then 0 else 1 end as fe_flag_w02,
                case when coalesce(b.fe_post_w03,0) <1 then 0 else 1 end as fe_flag_w03,
                case when coalesce(b.fe_post_w04,0) <1 then 0 else 1 end as fe_flag_w04,
                case when coalesce(b.fe_post_w05,0) <1 then 0 else 1 end as fe_flag_w05,
                case when coalesce(b.fe_post_w06,0) <1 then 0 else 1 end as fe_flag_w06,
                case when coalesce(b.fe_post_w07,0) <1 then 0 else 1 end as fe_flag_w07,
                case when coalesce(b.fe_post_w08,0) <1 then 0 else 1 end as fe_flag_w08,
                case when coalesce(b.fe_post_w09,0) <1 then 0 else 1 end as fe_flag_w09,
                case when coalesce(b.fe_post_w10,0) <1 then 0 else 1 end as fe_flag_w10

            from 
                dormant_customer a
            left join
                (
                select 
                    customerid,
                    -- sum(ao_sessions_unique_daily) ao,
                    -- sum(fe_sessions_unique_daily) fe,
                    -- sum(rr_sessions_unique_daily) rr,
                    -- sum(net_rides_daily) nr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then ao_sessions_unique_daily end) as ao,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w01,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w01,
                    
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w01

                from 
                    datasets.customer_rf_daily_kpi 
                where 
                    date(day) > date('{base_week}') 
                    and date(day) <= date('{base_week}') + interval '70' day
                    and service_name in ('Link','Auto', 'CabEconomy')
                    and city in ('{city_name}')
                    and customerid  = '6288c6694281e53bbe33d133'
                    -- IN ('63058bc3ca3c325e3991febd', '650c1afb1213a91bc9edf1a0', '665c1b793983e81f8c129b31', '63db6fe7beabc2a4ef03caa4', '654c65e15f58b4d23f3d6f8d', '66da8a60a1d537cc59aed940', '63a2121398760530cc3fe0f2', '63f8e3da959e097a43c4befb')
                    
                group by 1
                ) b
                on a.customer=b.customerid

        ),
    
        rr_data as (
        
            select 
                fe.*
            from
                (
                select 
                    aa.*, 
                    bb.rr_LT, 
                    bb.rapido_age,
                    bb.rapido_age_seg
                from
                    (
                    select 
                        customerid,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w13,
                        sum(case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w12,
                        sum(case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w11,
                        sum(case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w10,
                        sum(case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w09,
                        sum(case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w08,
                        sum(case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w07,
                        sum(case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w06,
                        sum(case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w05,
                        sum(case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w04,
                        sum(case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w03,
                        sum(case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w02,
                        sum(case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w01,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_91days,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then day end) as active_rr_days_w13,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then day end) as active_rr_days_w12,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then day end) as active_rr_days_w11,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then day end) as active_rr_days_w10,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then day end) as active_rr_days_w09,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then day end) as active_rr_days_w08,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then day end) as active_rr_days_w07,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then day end) as active_rr_days_w06,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then day end) as active_rr_days_w05,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then day end) as active_rr_days_w04,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then day end) as active_rr_days_w03,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then day end) as active_rr_days_w02,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then day end) as active_rr_days_w01,
                        day((date('{run_date}') + interval '-01' day) - date(max(day))) as days_since_last_rr
                    from 
                        (
                        select 
                            customerid,day, 
                            max(fe_sessions_unique_daily) as fe_sessions_unique_daily,  
                            sum(rr_sessions_unique_daily) as rr_sessions_unique_daily
                        from 
                            datasets.customer_rf_daily_kpi a
                        join
                            base b
                            on a.customerid = b.customer
                            and 1 = 1 
                            and city = '{city_name}'
                            and rr_sessions_unique_daily >= 1 
                            and day >= date_format(date('{run_date}') + interval '-91' day , '%Y-%m-%d')
                            and day <= date_format(date('{run_date}') + interval '-1' day , '%Y-%m-%d')
                            and service_name in ('Link','Auto', 'CabEconomy')

                            and customerid = '6288c6694281e53bbe33d133'
                            -- IN ('63058bc3ca3c325e3991febd', '650c1afb1213a91bc9edf1a0', '665c1b793983e81f8c129b31', '63db6fe7beabc2a4ef03caa4', '654c65e15f58b4d23f3d6f8d', '66da8a60a1d537cc59aed940', '63a2121398760530cc3fe0f2', '63f8e3da959e097a43c4befb')
                            
                        group by customerid, day
                        )
                    group by customerid
                    ) aa 
                left join 
                    (
                    select 
                        customerid, 
                        sum(rr_sessions_unique_daily) as rr_LT , 
                        day((date('{run_date}') + interval '-01' day) - date(min(day))) as rapido_age,
                        ceil(cast(day((date('{run_date}') + interval '-01' day) - date(min(day))) as real)/10)*10 rapido_age_seg
                    from 
                        datasets.customer_rf_daily_kpi a
                    join
                        base b
                        on a.customerid=b.customer
                        and rr_sessions_unique_daily >= 1 
                        and city = '{city_name}'

                        and customerid  = '6288c6694281e53bbe33d133'
                        -- IN ('63058bc3ca3c325e3991febd', '650c1afb1213a91bc9edf1a0', '665c1b793983e81f8c129b31', '63db6fe7beabc2a4ef03caa4', '654c65e15f58b4d23f3d6f8d', '66da8a60a1d537cc59aed940', '63a2121398760530cc3fe0f2', '63f8e3da959e097a43c4befb')
                        
                    group by 1
                    ) bb
                    on aa.customerid = bb.customerid
                ) fe 
        )
        
        select 
            a.*, b.*,
            '{run_date}' as run_date
        from 
            base a
        left join
            rr_data b
            on a.customer=b.customerid
    """

    try:
        data = pd.read_sql(rr_data_query, connection3)
    except:
        data = pd.read_sql(rr_data_query, connection4)
        
    
    ## Data fetched successfully
    print(data.shape)


    ## .mask(data[columns_to_replace] == 0, pd.NA) replaces all 0 values in these columns with pd.NA.
    columns_to_replace = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11','active_rr_days_w10', 'active_rr_days_w09',
                          'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05', 'active_rr_days_w04',
                          'active_rr_days_w03', 'active_rr_days_w02', 'active_rr_days_w01']
    data[columns_to_replace] = data[columns_to_replace].mask(data[columns_to_replace] == 0, pd.NA)
    

    ## Unused
    ## RR-Normalization = Total no. of RR/Active days
    """
    data['rr_norm_w13'] = data.rr_w13/data.active_rr_days_w13
    data['rr_norm_w12'] = data.rr_w12/data.active_rr_days_w12
    data['rr_norm_w11'] = data.rr_w11/data.active_rr_days_w11
    data['rr_norm_w10'] = data.rr_w10/data.active_rr_days_w10
    data['rr_norm_w09'] = data.rr_w09/data.active_rr_days_w09
    data['rr_norm_w08'] = data.rr_w08/data.active_rr_days_w08
    data['rr_norm_w07'] = data.rr_w07/data.active_rr_days_w07
    data['rr_norm_w06'] = data.rr_w06/data.active_rr_days_w06
    data['rr_norm_w05'] = data.rr_w05/data.active_rr_days_w05
    data['rr_norm_w04'] = data.rr_w04/data.active_rr_days_w04
    data['rr_norm_w03'] = data.rr_w03/data.active_rr_days_w03
    data['rr_norm_w02'] = data.rr_w02/data.active_rr_days_w02
    data['rr_norm_w01'] = data.rr_w01/data.active_rr_days_w01
    """
    
    raw_customer_data = data[data['customerid']!=0]
    back_up = raw_customer_data.copy() ## With NA as value
    fe_data2 = raw_customer_data.fillna(0)
    fe_data = fe_data2[fe_data2['customerid']!=0] ## With 0 as value
    
    
    ## Consistency/Regularity Threshold 
    ## RR Regularity → Count of active weeks in last 13 weeks/ (13 or Rapido age in weeks if a customer is less than 13 weeks old)
    ## Active Days Mean → Total active days in last 13 weeks/ count of active weeks in last 13 weeks
    
    ## [OLD version - Unused] 
    """
    def regularity_decider(x,active_mean):
    
        if x<=0.3:
            return '05_rare_need'
        elif x>0.3 and x<=0.5:
            return '04_monthly'
        elif x>0.5 and x<=0.75:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'
        # return 'weekly'
    """

    ## [Relaxed threshold]
    def regularity_decider_v2(x,active_mean):
        
        if x<=0.04:
            return '05_rare_need'
        elif x>0.04 and x<=0.08:
            return '04_monthly'
        elif x>0.08 and x<=0.15:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'


    ## Intent Threshold
    ## Intent → Total RR in last 13 weeks/ No. of Active weeks
    def intent_decider(x):
        
        if x<=2:
            return '03_Low'
        elif x>2 and x<=5:
            return '02_Medium'
        else:
            return '01_High'
    

    ## Consideration signal
    def consideration_signal_creator(x,y,active_mean):
        
        if y<=1.5:
            if x<=0.3:
                return 'rarely_1_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_1_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_1_rr'
            else:
                return 'weekly_1_rr'
        else:
            if x<=0.3:
                return 'rarely_2+_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_2+_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_2+_rr'
            else:
                return 'weekly_2+_rr'
                
                # Dead-code
                # if active_mean>2.6:
                #     return 'daily'
                # else:
                #     return 'weekly'


    ## Correction factor
    def correction_factor(rr_weight,rapido_age):
        
        if rapido_age >= 90:
            week_inactive = 0
            multiplier = 1

        elif rapido_age == 0:
            week_inactive = (90-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- np.log(week_inactive*(week_inactive+1)/2)))

        # 7,14,21,28,...
        elif rapido_age%7 == 0:
            week_inactive = (90-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- np.log(week_inactive*(week_inactive+1)/2)))    
            
        else:
            week_inactive = (90-rapido_age)//7
            multiplier = (rr_weight.sum()/(rr_weight.sum()- np.log(week_inactive*(week_inactive+1)/2))) 
        
        return (multiplier, 
                week_inactive, 
                (rr_weight.sum()- np.log(week_inactive*(week_inactive+1)/2)), 
                ((rr_weight*rr_weight).sum() - np.log(week_inactive*(week_inactive+1)*(2*week_inactive+1)/6))
               )

    
    ## Recency trend
    def recency_type(days_since_last_fe):
        
        if days_since_last_fe <= 6: 
            return('Recent')
        elif days_since_last_fe <= 20:
            return('Stationary')
        else:
            return('Dormant')

    ## Intent trend
    def intent_trend_type(intent_trend):
        
        if intent_trend <= -0.1:
            return('03_Declining')
        elif intent_trend <= 0.1:
            return('02_Stable')
        else:
            return('01_Increasing')

    # Dead-code
    # def custom_func(mean_intent,rr_trend):
    #     if mean_intent=='High':
    #         return rr_trend
    #     elif mean_intent=='Low' and rr_trend=='01_Increasing':
    #         return '01_Increasing'
    #     else:
    #         return '02_Stable'

    
    
    ## Creating Final signal
    def create_L3_signals(fe_data, back_up): 
        
        print('Creating L3 Signal')
        rr_weight = np.log(np.array([3,4,5,6,7,8,9,10,11,12,13,14,15]))

        ## sum (rr_sessions_unique_daily)
        fe_cols = ['rr_w13', 'rr_w12', 'rr_w11', 'rr_w10', 'rr_w09','rr_w08', 'rr_w07', 
                   'rr_w06','rr_w05', 'rr_w04', 'rr_w03', 'rr_w02','rr_w01']   
        ## count (rr_days)
        activedays_cols = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11', 'active_rr_days_w10', 
                           'active_rr_days_w09','active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06',
                           'active_rr_days_w05', 'active_rr_days_w04', 'active_rr_days_w03','active_rr_days_w02', 
                           'active_rr_days_w01'] 
        
        ## Unused
        # norm_intent=['rr_norm_w13', 'rr_norm_w12', 'rr_norm_w11', 'rr_norm_w10','rr_norm_w09', 'rr_norm_w08', 'rr_norm_w07', 
        #              'rr_norm_w06','rr_norm_w05', 'rr_norm_w04', 'rr_norm_w03', 'rr_norm_w02','rr_norm_w01']
        # fe_data['fe2rr_91days'] = fe_data.apply(lambda x: (x['rr_91days']/x['fe_91days']) if x['fe_91days'] > 0 else 0.5,axis=1)
        
        
        print('Calctn-1')
        fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight,x)[0])
        ## Unused
        # fe_data['rr_need'] = np.array(fe_data[fe_cols]).max(axis=1)
        # fe_data['no_week_rr_done'] =  back_up[fe_cols].count(axis=1)
        # fe_data['%week_rr_done'] = fe_data.apply(lambda x: x['no_week_rr_done']*(1/(13 - correction_factor(rr_weight,x['rapido_age'])[1])), axis = 1 )
        fe_data['rr_mean_intent'] =  back_up[fe_cols].mean(axis=1) ## Check logic once

        ## Unused
        # fe_data['rr_mean_intent_norm'] =  back_up[norm_intent].mean(axis=1)
        # fe_data['rr_mean_intent_type'] = fe_data['rr_mean_intent'].apply(lambda x: 'Low' if x <= 3 else 'High')
        fe_data['rr_mean_intent_final'] = fe_data['rr_mean_intent'].apply(lambda x: intent_decider(x))

        # Dead-code
        # fe_data['rr_cov'] =  (np.sqrt(back_up[fe_cols].var(axis=1))/back_up[fe_cols].mean(axis=1))
        # fe_data['rr_volatility_type'] = fe_data.apply(lambda x: 'Low' if x['rr_cov'] <= 0.4 else 'High', axis = 1)
        # fe_data['rr_potential'] = fe_data['rr_need'] - fe_data['rr_mean_intent']
        # fe_data['rr_potential_per'] = (fe_data['rr_need'] - fe_data['rr_mean_intent'])/fe_data['rr_mean_intent']
        # fe_data['rr_potential_per_type'] = fe_data['rr_potential_per'].apply(lambda x: 'High' if x >= 0.5 else 'Low')
        
        ## Unused
        # fe_data['rr_trend_n'] = (((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis=1)) * (np.array(fe_data[fe_cols])*rr_weight).sum(axis=1))  - (np.array(fe_data[fe_cols]).sum(axis=1) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis=1))) 
        # fe_data['rr_trend_d'] = (((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis = 1)) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[3],axis =1)) -(fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis =1))**2 )
        # fe_data['rr_trend'] = fe_data['rr_trend_n']/fe_data['rr_trend_d']
        # fe_data['rr_trend'] = fe_data['rr_trend'].fillna(0)
        # fe_data['rr_intent_trend_type'] = fe_data['rr_trend'].apply(lambda x: intent_trend_type(x))
        

        # Notes
        # fe.ge(1.0): This method compares each element in the DataFrame fe to 1.0. 
        # (True if the element is greater than or equal to 1.0, False otherwise).
        # .astype(int): This method converts the boolean values to integers (1 for True and 0 for False).
        fe = fe_data[fe_cols]
        fe = fe.ge(1.0).astype(int)
        
        
        # This multiplies the sum of the weighted values by 1/91 to normalize the result.
        # rr_w13 to rr_w01 || rr_weight 1 to 13
        fe_data['rr_regularity'] = (1/90)*(np.array(fe)*rr_weight).sum(axis=1)
        fe_data['rr_regularity_scaled'] = fe_data['rr_regularity']*fe_data['rr_intent_multiplier']
        
        # Unused
        # fe_data['rr_regularity_type'] = fe_data['rr_regularity_scaled'].apply(lambda x: 'Low' if x <= 0.3 else 'High')
        # fe_data['rr_recency_type'] = fe_data['days_since_last_rr'].apply(lambda x: recency_type(x))
        
        fe_data['active_days_mean'] =  back_up[activedays_cols].mean(axis=1) ## Check logic once
        # fe_data['rr_regularity_final_old'] = fe_data.apply(lambda x: regularity_decider(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        
        print('Calctn-2')
        fe_data['rr_regularity_final'] = fe_data.apply(lambda x: regularity_decider_v2(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        fe_data['reg_intent'] = fe_data.rr_regularity_final+'_'+fe_data.rr_mean_intent_final
        
        # print('Calctn-3')
        # fe_data['consideration_final'] = fe_data.apply(lambda x: consideration_signal_creator(x['rr_regularity_scaled'],x['rr_mean_intent'],x['active_days_mean']),axis=1)
        # fe_data['rr_intent_trend_type_v2']= fe_data.apply(lambda x: custom_func(x['rr_mean_intent_type'],x['rr_intent_trend_type']),axis=1)

        return(fe_data)

    
    print('UDF-Created Successfully')
    
    
    L3_signals = create_L3_signals(fe_data,back_up)
    L3_signals['city'] = city_name
    
    final_data = L3_signals.copy()
    
    # final_data = final_data.add_suffix('_nov22')
    # final_data['target']=np.where(final_data.next_lts=='DORMANT',1,0)
    
    return final_data    

## Signal Generation

In [5]:
concatenated_df=pd.DataFrame()

for i in date_list:
    
    print(i)
    tdf = consideration_query(i, city, base_week)
    
    print(tdf.shape[0])
    # concatenated_df=concatenated_df.append(tdf).reset_index(drop=True)
    concatenated_df=pd.concat([concatenated_df, tdf]).reset_index(drop=True)

2024-09-20
(1, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
1


In [6]:
concatenated_df.rr_regularity_scaled.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00    0.250063
0.10    0.250063
0.20    0.250063
0.30    0.250063
0.40    0.250063
0.50    0.250063
0.60    0.250063
0.70    0.250063
0.80    0.250063
0.90    0.250063
0.95    0.250063
0.99    0.250063
1.00    0.250063
Name: rr_regularity_scaled, dtype: float64

In [7]:
concatenated_df.active_days_mean.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00    3.545455
0.10    3.545455
0.20    3.545455
0.30    3.545455
0.40    3.545455
0.50    3.545455
0.60    3.545455
0.70    3.545455
0.80    3.545455
0.90    3.545455
0.95    3.545455
0.99    3.545455
1.00    3.545455
Name: active_days_mean, dtype: float64

In [8]:
concatenated_df.rr_mean_intent.quantile([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])

0.00    7.818182
0.10    7.818182
0.20    7.818182
0.30    7.818182
0.40    7.818182
0.50    7.818182
0.60    7.818182
0.70    7.818182
0.80    7.818182
0.90    7.818182
0.95    7.818182
0.99    7.818182
1.00    7.818182
Name: rr_mean_intent, dtype: object

In [9]:
concatenated_df['reg_intent'] = concatenated_df.rr_regularity_final + '_' + concatenated_df.rr_mean_intent_final
base_df = concatenated_df[concatenated_df.run_date == base_week].reset_index(drop=True)

In [10]:
concatenated_df.head(5)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,6288c6694281e53bbe33d133,852,860.0,2024-09-19,2022-05-22,COMMITTED,COMMITTED,HIGH_INCOME,234,RECENT,12,Bangalore,High,Stable,High,High,Recent,High,MALE,PS,PS,ONLY_LINK,SHORT_DISTANCE,BOTH_PEAK,WEEKDAY,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,6288c6694281e53bbe33d133,6,8,9,7,6,14,4,0,0,9,10,7,6,86,95,3,4,5,3,3,4,2,0.0,0.0,3,5,3,4,0,229,852,860.0,2024-09-20,1,7.818182,01_High,0.250063,0.250063,3.545455,01_daily,01_daily_01_High,Bangalore


In [11]:
concatenated_df[['customer', 'rr_mean_intent', 'rr_regularity', 'rr_regularity_scaled', 'active_days_mean', 'rr_mean_intent_final', 'rr_regularity_final', 'reg_intent']]

,customer,rr_mean_intent,rr_regularity,rr_regularity_scaled,active_days_mean,rr_mean_intent_final,rr_regularity_final,reg_intent
0,6288c6694281e53bbe33d133,7.818182,0.250063,0.250063,3.545455,01_High,01_daily,01_daily_01_High


In [12]:
concatenated_df[concatenated_df['customer'] == '6288c6694281e53bbe33d133']\
[['customer', 'rr_mean_intent', 'rr_regularity', 'rr_regularity_scaled', 'active_days_mean', 'rr_mean_intent_final', 'rr_regularity_final', 'reg_intent']]

,customer,rr_mean_intent,rr_regularity,rr_regularity_scaled,active_days_mean,rr_mean_intent_final,rr_regularity_final,reg_intent
0,6288c6694281e53bbe33d133,7.818182,0.250063,0.250063,3.545455,01_High,01_daily,01_daily_01_High


In [12]:
concatenated_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,5dd787c061ebec69bf94e3af,873,880.0,2024-04-07,2021-11-16,SUSTENANCE,DORMANT,UNKNOWN,77,RECENT,2,Chennai,Low,Stable,High,Low,Recent,High,MALE,UNKNOWN,UNKNOWN,ONLY_LINK,UNKNOWN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5dd787c061ebec69bf94e3af,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,56.0,872.0,880.0,2024-04-07,1.00000,1.000000,03_Low,0.028186,0.028186,1.0,05_rare_need,05_rare_need_03_Low,Chennai
1,646da257d87110f93d5f5481,319,320.0,2024-04-07,2023-05-24,COMMITTED,COMMITTED,MEDIUM_INCOME,57,RECENT,16,Chennai,High,Increasing,High,High,Recent,High,FEMALE,PS,NPS,ONLY_LINK,SHORT_DISTANCE,MORNING_PEAK,WEEKDAY,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,646da257d87110f93d5f5481,0.0,0.0,0.0,0.0,2.0,1.0,2.0,7.0,6.0,14.0,6.0,8.0,5.0,51.0,56.0,0.0,0.0,0.0,0.0,2.0,1.0,2.0,4.0,6.0,6.0,5.0,5.0,5.0,1.0,191.0,318.0,320.0,2024-04-07,1.00000,5.666667,01_High,0.212902,0.212902,4.0,01_daily,01_daily_01_High,Chennai
2,6585c073e6293df49a4e4e9a,57,60.0,2024-04-06,2024-02-10,COMMITTED,SUSTENANCE,HIGH_INCOME,10,RECENT,3,Chennai,Low,Stable,High,High,Recent,High,FEMALE,NPS,UNKNOWN,ONLY_LINK,SHORT_DISTANCE,UNKNOWN,UNKNOWN,0,0,0,0,0,0,0,0,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,6585c073e6293df49a4e4e9a,0.0,0.0,0.0,0.0,0.0,2.0,3.0,0.0,1.0,0.0,2.0,0.0,0.0,8.0,12.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0,0.0,1.0,0.0,2.0,0.0,0.0,15.0,13.0,56.0,60.0,2024-04-07,1.11371,2.000000,03_Low,0.091569,0.101981,2.0,03_bi_weekly,03_bi_weekly_03_Low,Chennai


In [13]:
base_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,5dd787c061ebec69bf94e3af,873,880.0,2024-04-07,2021-11-16,SUSTENANCE,DORMANT,UNKNOWN,77,RECENT,2,Chennai,Low,Stable,High,Low,Recent,High,MALE,UNKNOWN,UNKNOWN,ONLY_LINK,UNKNOWN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5dd787c061ebec69bf94e3af,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,56.0,872.0,880.0,2024-04-07,1.00000,1.000000,03_Low,0.028186,0.028186,1.0,05_rare_need,05_rare_need_03_Low,Chennai
1,646da257d87110f93d5f5481,319,320.0,2024-04-07,2023-05-24,COMMITTED,COMMITTED,MEDIUM_INCOME,57,RECENT,16,Chennai,High,Increasing,High,High,Recent,High,FEMALE,PS,NPS,ONLY_LINK,SHORT_DISTANCE,MORNING_PEAK,WEEKDAY,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,646da257d87110f93d5f5481,0.0,0.0,0.0,0.0,2.0,1.0,2.0,7.0,6.0,14.0,6.0,8.0,5.0,51.0,56.0,0.0,0.0,0.0,0.0,2.0,1.0,2.0,4.0,6.0,6.0,5.0,5.0,5.0,1.0,191.0,318.0,320.0,2024-04-07,1.00000,5.666667,01_High,0.212902,0.212902,4.0,01_daily,01_daily_01_High,Chennai
2,6585c073e6293df49a4e4e9a,57,60.0,2024-04-06,2024-02-10,COMMITTED,SUSTENANCE,HIGH_INCOME,10,RECENT,3,Chennai,Low,Stable,High,High,Recent,High,FEMALE,NPS,UNKNOWN,ONLY_LINK,SHORT_DISTANCE,UNKNOWN,UNKNOWN,0,0,0,0,0,0,0,0,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,0,0,0,0,1,1,1,1,1,1,6585c073e6293df49a4e4e9a,0.0,0.0,0.0,0.0,0.0,2.0,3.0,0.0,1.0,0.0,2.0,0.0,0.0,8.0,12.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0,0.0,1.0,0.0,2.0,0.0,0.0,15.0,13.0,56.0,60.0,2024-04-07,1.11371,2.000000,03_Low,0.091569,0.101981,2.0,03_bi_weekly,03_bi_weekly_03_Low,Chennai


In [14]:
concatenated_df.to_csv('rr_concatenated_df_log.csv', index=False)
base_df.to_csv('rr_base_df_log.csv', index=False)

# concatenated_df = pd.read_csv('concatenated_df.csv')
# base_df = pd.read_csv('base_df.csv')

## Stability Function

In [15]:
def stability_func(df,base_week):
    
    df_base_week = df[df.run_date == base_week][['customer','run_date','rr_regularity_final','rr_mean_intent_final','reg_intent']]\
                                                    .reset_index(drop=True)
    ranking = {'05_rare_need' : 1,
               '04_monthly' : 2,
               '03_bi_weekly' : 3,
               '02_weekly' : 4,
               '01_daily' : 5
              }

    intent_ranking = {'03_Low' : 1,
                      '02_Medium' : 2,
                      '01_High' : 3
                     }

    final_rank = {'05_rare_need_03_Low' : 1,
                  '05_rare_need_02_Medium' : 2,
                  '05_rare_need_01_High' : 3,
                  '04_monthly_03_Low' : 4,
                  '04_monthly_02_Medium' : 5,
                  '04_monthly_01_High' : 6,
                  '03_bi_weekly_03_Low' : 7,
                  '03_bi_weekly_02_Medium' : 8,
                  '03_bi_weekly_01_High' : 9,
                  '02_weekly_03_Low' : 10,
                  '02_weekly_02_Medium' : 11,
                  '02_weekly_01_High' : 12,
                  '01_daily_02_Medium' : 13,
                  '01_daily_01_High' : 14
                 }    

    df['reg_rank'] = df['rr_regularity_final'].map(ranking)
    df['int_rank'] = df['rr_mean_intent_final'].map(intent_ranking)
    df['final_rank'] = df['reg_intent'].map(final_rank)

    
    
    ## Regularity Stabillity
    pvt_df = pd.pivot_table(df, values='reg_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df.columns
    dropped_reg = pvt_df.iloc[:, 1:].lt(pvt_df[base_week], axis=0)

    # Display the result
    print(dropped_reg.head())
    dropped_reg.head()
    first_dropped_week = dropped_reg.idxmax(axis=1)

    pvt_df['first_dropped_week'] = first_dropped_week.where(dropped_reg.any(axis=1), 'Maintained/Upgraded')

    pvt_df.reset_index(inplace=True)
    pvt_df.head()

    df_base_week_stability = pvt_df.merge(df_base_week, on=['customer'], how='inner')

    reg_stab = df_base_week_stability\
                        .groupby(by=['run_date','rr_regularity_final','first_dropped_week'], as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_regularity_final',], 
                                     columns='first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['rr_regularity_final'],as_index=False)\
                                   .agg(base=('customer','nunique')), on=['rr_regularity_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/regularity_stability.csv',index=False)

    
    
    
    ## Intent Stabillity
    pvt_df_int=pd.pivot_table(df, values='int_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_int.columns
    dropped_int = pvt_df_int.iloc[:, 1:].lt(pvt_df_int[base_week], axis=0)

    # Display the result
    print(dropped_int.head())
    dropped_int.head()
    int_first_dropped_week = dropped_int.idxmax(axis=1)

    pvt_df_int['int_first_dropped_week'] = int_first_dropped_week.where(dropped_int.any(axis=1), 'Maintained/Upgraded')

    pvt_df_int.reset_index(inplace=True)
    pvt_df_int.head()

    df_base_week_int_stability = pvt_df_int.merge(df_base_week, on=['customer'], how='inner')

    int_stab = df_base_week_int_stability\
                        .groupby(by=['run_date','rr_mean_intent_final','int_first_dropped_week'],as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_mean_intent_final',], 
                                     columns='int_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                    .groupby(by=['rr_mean_intent_final'], as_index=False)\
                                    .agg(base=('customer','nunique')), on=['rr_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/intent_stability.csv',index=False)

    
    
    
    ## Final Stabillity
    pvt_df_final=pd.pivot_table(df, values='final_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_final.columns
    dropped_final = pvt_df_final.iloc[:, 1:].lt(pvt_df_final[base_week], axis=0)

    
    # Display the result
    print(dropped_final.head())
    dropped_final.head()
    final_first_dropped_week = dropped_final.idxmax(axis=1)

    pvt_df_final['final_first_dropped_week'] = final_first_dropped_week.where(dropped_final.any(axis=1), 'Maintained/Upgraded')

    pvt_df_final.reset_index(inplace=True)
    pvt_df_final.head()

    df_base_week_final_stability = pvt_df_final.merge(df_base_week, on=['customer'], how='inner')

    final_stab = df_base_week_final_stability\
                        .groupby(by=['run_date','rr_regularity_final','rr_mean_intent_final','final_first_dropped_week'],
                                 as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_regularity_final','rr_mean_intent_final'], 
                                     columns='final_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['rr_regularity_final','rr_mean_intent_final'], as_index=False)\
                                   .agg(base=('customer','nunique')), on=['rr_regularity_final','rr_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/final_stability.csv',index=False)
    
    return reg_stab, int_stab, final_stab

In [16]:
reg_df, int_df, final_seg_df = stability_func(concatenated_df, base_week)

run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False       False       False       False
2              False       False       False       False       False
3              False        True       False        True        True
4              False       False       False       False       False
run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False       False       False       False
2              False       False       False       False       False
3              False       False        True        True        True
4              False       False       False       False       False
run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False  

In [17]:
# complete flow
def stab_func_2(df, base_week, base_df, cons_type):
    
    # base_df=df[df.run_date==base_week].reset_index(drop=True)
    fdf1 = base_df.groupby(by=['city', 'run_date', cons_type], as_index=False).agg(base=('customer','nunique'))
    j = 0
    
    for i in date_list:
        
        j += 1
        if i != base_week:
            
            tdf = df[df.run_date==base_week][['run_date', 'customer', cons_type]]\
                                                .merge(df[df.run_date == i][['customer', cons_type]],on=['customer'],how='left')\
                                                .pivot_table(values='customer', 
                                                             index=['run_date',cons_type+'_x'], 
                                                             columns=cons_type+'_y', aggfunc='nunique')\
                                                .reset_index()\
                                                .fillna(0)
            
            # print(list(tdf))
            items_to_remove = ['run_date', cons_type+'_x']
            filtered_list = [x for x in list(tdf) if x not in items_to_remove]
            tdf.columns = ['run_date', cons_type]+[ j + '_' + i for j in filtered_list]
            fdf1 = fdf1.merge(tdf, on=['run_date', cons_type], how='inner')
    
    return fdf1
        
        

In [18]:
#cons_type: 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
finaldf = stab_func_2(concatenated_df, base_week, base_df, 'reg_intent')

In [19]:
finaldf

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14,01_daily_01_High_2024-04-21,01_daily_02_Medium_2024-04-21,02_weekly_01_High_2024-04-21,02_weekly_02_Medium_2024-04-21,02_weekly_03_Low_2024-04-21,03_bi_weekly_01_High_2024-04-21,03_bi_weekly_02_Medium_2024-04-21,03_bi_weekly_03_Low_2024-04-21,04_monthly_01_High_2024-04-21,04_monthly_02_Medium_2024-04-21,04_monthly_03_Low_2024-04-21,05_rare_need_01_High_2024-04-21,05_rare_need_02_Medium_2024-04-21,05_rare_need_03_Low_2024-04-21,01_daily_01_High_2024-04-28,01_daily_02_Medium_2024-04-28,02_weekly_01_High_2024-04-28,02_weekly_02_Medium_2024-04-28,02_weekly_03_Low_2024-04-28,03_bi_weekly_01_High_2024-04-28,03_bi_weekly_02_Medium_2024-04-28,03_bi_weekly_03_Low_2024-04-28,04_monthly_01_High_2024-04-28,04_monthly_02_Medium_2024-04-28,04_monthly_03_Low_2024-04-28,05_rare_need_01_High_2024-04-28,05_rare_need_02_Medium_2024-04-28,05_rare_need_03_Low_2024-04-28,01_daily_01_High_2024-05-05,01_daily_02_Medium_2024-05-05,02_weekly_01_High_2024-05-05,02_weekly_02_Medium_2024-05-05,02_weekly_03_Low_2024-05-05,03_bi_weekly_01_High_2024-05-05,03_bi_weekly_02_Medium_2024-05-05,03_bi_weekly_03_Low_2024-05-05,04_monthly_01_High_2024-05-05,04_monthly_02_Medium_2024-05-05,04_monthly_03_Low_2024-05-05,05_rare_need_01_High_2024-05-05,05_rare_need_02_Medium_2024-05-05,05_rare_need_03_Low_2024-05-05
0,Chennai,2024-04-07,01_daily_01_High,33958,31422.0,1639.0,162.0,223.0,0.0,229.0,25.0,0.0,0.0,0.0,0.0,251.0,7.0,0.0,30590.0,2037.0,217.0,396.0,0.0,338.0,61.0,0.0,0.0,0.0,0.0,285.0,34.0,0.0,29559.0,2471.0,230.0,656.0,0.0,539.0,139.0,0.0,0.0,0.0,0.0,341.0,23.0,0.0,28208.0,2925.0,233.0,1006.0,2.0,944.0,255.0,6.0,12.0,7.0,1.0,318.0,41.0,0.0
1,Chennai,2024-04-07,01_daily_02_Medium,23566,1449.0,19229.0,1.0,2385.0,0.0,10.0,336.0,0.0,0.0,0.0,0.0,8.0,148.0,0.0,2415.0,17295.0,8.0,3009.0,3.0,29.0,536.0,0.0,0.0,0.0,0.0,15.0,256.0,0.0,3347.0,15438.0,13.0,3618.0,13.0,68.0,794.0,1.0,0.0,0.0,0.0,23.0,251.0,0.0,3849.0,13730.0,12.0,4284.0,33.0,124.0,1237.0,14.0,5.0,21.0,2.0,26.0,229.0,0.0
2,Chennai,2024-04-07,02_weekly_01_High,1072,180.0,3.0,604.0,247.0,0.0,21.0,8.0,0.0,0.0,0.0,0.0,6.0,3.0,0.0,244.0,5.0,468.0,300.0,1.0,32.0,9.0,0.0,0.0,0.0,0.0,10.0,3.0,0.0,269.0,12.0,388.0,318.0,1.0,43.0,28.0,0.0,0.0,0.0,0.0,10.0,3.0,0.0,263.0,18.0,300.0,348.0,1.0,67.0,50.0,2.0,0.0,0.0,0.0,18.0,5.0,0.0
3,Chennai,2024-04-07,02_weekly_02_Medium,80709,256.0,2507.0,248.0,69279.0,2954.0,4.0,4457.0,261.0,0.0,0.0,0.0,3.0,732.0,8.0,618.0,3854.0,314.0,64279.0,3850.0,23.0,6113.0,661.0,0.0,0.0,0.0,7.0,961.0,29.0,1120.0,4998.0,351.0,59387.0,4687.0,42.0,7837.0,1365.0,0.0,0.0,0.0,11.0,873.0,38.0,1589.0,5617.0,402.0,54910.0,5158.0,88.0,9750.0,2116.0,2.0,105.0,46.0,20.0,864.0,42.0
4,Chennai,2024-04-07,02_weekly_03_Low,35915,0.0,2.0,0.0,3387.0,27014.0,0.0,130.0,5093.0,0.0,0.0,0.0,0.0,14.0,275.0,1.0,11.0,0.0,5098.0,23556.0,0.0,384.0,6349.0,0.0,0.0,0.0,0.0,63.0,453.0,2.0,39.0,1.0,6470.0,20339.0,0.0,752.0,7826.0,0.0,0.0,0.0,0.0,63.0,423.0,7.0,85.0,3.0,7298.0,17403.0,0.0,1181.0,9249.0,0.0,11.0,110.0,0.0,116.0,452.0
5,Chennai,2024-04-07,03_bi_weekly_01_High,4315,887.0,89.0,68.0,62.0,0.0,2590.0,440.0,1.0,135.0,14.0,0.0,29.0,0.0,0.0,1437.0,168.0,85.0,135.0,0.0,1764.0,490.0,3.0,175.0,32.0,0.0,22.0,4.0,0.0,1635.0,210.0,73.0,208.0,0.0,1368.0,525.0,12.0,181.0,55.0,3.0,29.0,16.0,0.0,1669.0,269.0,92.0,298.0,1.0,1063.0,500.0,28.0,230.0,72.0,15.0,48.0,28.0,2.0
6,Chennai,2024-04-07,03_bi_weekly_02_Medium,51596,131.0,959.0,34.0,7093.0,626.0,406.0,35672.0,3541.0,8.0,2411.0,200.0,2.0,488.0,25.0,428.0,1607.0,57.0,9711.0,1141.0,403.0,29376.0,4740.0,18.0,3046.0,496.0,14.0,522.0,

In [20]:
base_df\
.groupby(by=['city_name','run_date','reg_intent'],as_index=False)\
.agg(
    user_count=('customer','nunique'),
    fe_retention_w01=('fe_flag_w01','sum'),
    fe_retention_w02=('fe_flag_w02','sum'),
    fe_retention_w03=('fe_flag_w03','sum'),
    fe_retention_w04=('fe_flag_w04','sum'),
    fe_retention_w05=('fe_flag_w05','sum'),
    fe_retention_w06=('fe_flag_w06','sum'),
    fe_retention_w07=('fe_flag_w07','sum'),
    fe_retention_w08=('fe_flag_w08','sum'),
    fe_retention_w09=('fe_flag_w09','sum'),
    fe_retention_w10=('fe_flag_w10','sum'),
    
    rr_retention_w01=('rr_flag_w01','sum'),
    rr_retention_w02=('rr_flag_w02','sum'),
    rr_retention_w03=('rr_flag_w03','sum'),
    rr_retention_w04=('rr_flag_w04','sum'),
    rr_retention_w05=('rr_flag_w05','sum'),
    rr_retention_w06=('rr_flag_w06','sum'),
    rr_retention_w07=('rr_flag_w07','sum'),
    rr_retention_w08=('rr_flag_w08','sum'),
    rr_retention_w09=('rr_flag_w09','sum'),
    rr_retention_w10=('rr_flag_w10','sum'),
        
    nr_retention_w01=('nr_flag_w01','sum'),
    nr_retention_w02=('nr_flag_w02','sum'),
    nr_retention_w03=('nr_flag_w03','sum'),
    nr_retention_w04=('nr_flag_w04','sum'),
    nr_retention_w05=('nr_flag_w05','sum'),
    nr_retention_w06=('nr_flag_w06','sum'),
    nr_retention_w07=('nr_flag_w07','sum'),
    nr_retention_w08=('nr_flag_w08','sum'),
    nr_retention_w09=('nr_flag_w09','sum'),
    nr_retention_w10=('nr_flag_w10','sum')
    )


# .to_csv(f'{month}/output_files/reg_retention_10w_newlogic.csv',index=False)

#rr_mean_intent_final, rr_regularity_final, reg_intent

# gender_tag,
# taxi_income_segment,
# ps_tag_link,
# ps_tag_auto,
# taxi_service_affinity,
# taxi_distance_preference,
# taxi_time_of_day_preference,
# taxi_day_of_week_preference

,city_name,run_date,reg_intent,user_count,fe_retention_w01,fe_retention_w02,fe_retention_w03,fe_retention_w04,fe_retention_w05,fe_retention_w06,fe_retention_w07,fe_retention_w08,fe_retention_w09,fe_retention_w10,rr_retention_w01,rr_retention_w02,rr_retention_w03,rr_retention_w04,rr_retention_w05,rr_retention_w06,rr_retention_w07,rr_retention_w08,rr_retention_w09,rr_retention_w10,nr_retention_w01,nr_retention_w02,nr_retention_w03,nr_retention_w04,nr_retention_w05,nr_retention_w06,nr_retention_w07,nr_retention_w08,nr_retention_w09,nr_retention_w10
0,Chennai,2024-04-07,01_daily_01_High,33958,31338,32838,33229,33390,33489,33555,33590,33621,33656,33670,30757,32549,33056,33264,33383,33462,33504,33547,33589,33606,30099,32223,32856,33121,33273,33373,33433,33486,33538,33562
1,Chennai,2024-04-07,01_daily_02_Medium,23566,20831,22418,22894,23087,23161,23231,23277,23309,23340,23367,20128,22079,22680,22942,23048,23130,23188,23227,23262,23303,19245,21615,22409,22765,22925,23037,23111,23161,23197,23246
2,Chennai,2024-04-07,02_weekly_01_High,1072,882,987,1016,1034,1042,1045,1048,1052,1055,1057,845,970,1007,1026,1036,1041,1045,1049,1051,1055,806,944,987,1007,1020,1028,1035,1039,1040,1047
3,Chennai,2024-04-07,02_weekly_02_Medium,80709,63074,73584,76885,78202,78888,79267,79475,79650,79784,79874,58520,70937,75228,77075,78062,78621,78944,79212,79405,79563,54806,68104,73258,75674,76981,77781,78240,78603,78899,79131
4,Chennai,2024-04-07,02_weekly_03_Low,35915,23827,30544,33046,34077,34639,34941,35087,35219,35321,35413,21024,28388,31537,32965,33809,34279,34551,34780,34962,35095,19129,26480,30039,31816,32907,33530,33936,34273,34539,34713
5,Chennai,2024-04-07,03_bi_weekly_01_High,4315,3310,3635,3822,3898,3959,3999,4022,4046,4063,4079,3102,3519,3741,3841,3902,3945,3964,3995,4019,4037,2999,3446,3679,3797,3864,3907,3931,3970,3993,4014
6,Chennai,2024-04-07,03_bi_weekly_02_Medium,51596,32775,41272,44908,46835,47951,48637,49059,49401,49692,49893,28816,38086,42521,44941,46417,47344,47948,48418,48860,49164,26651,35762,40520,43278,45010,46139,46914,47522,48042,48436
7,Chennai,2024-04-07,03_bi_weekly_03_Low,86496,44353,62259,70700,75342,78143,79931,81103,81937,82689,83274,36394,54330,63973,69618,73357,75877,77540,78827,80057,80997,32308,49146,58961,65097,69409,72384,74480,76108,77524,78686
8,Chennai,2024-04-07,04_monthly_01_High,1796,1155,1308,1387,1429,1470,1490,1510,1533,1550,1560,1089,1241,1333,1379,1425,1448,1469,1494,1515,1528,1054,1204,1297,1348,1403,1429,1453,1478,1498,1513
9,Chennai,2024-04-07,04_monthly_02_Medium,18251,9186,12102,13600,14555,15213,15656,15936,16180,16372,16546,7703,10620,12259,13364,14152,14698,15060,15386,15666,15912,7117,9892,11515,12675,13529,14125,14524,14877,15181,15460


In [21]:
test=list(finaldf)

In [22]:
# test.remove([['city','01_daily_2023-11-11']])

In [23]:
test=['city']+[i+'_yo' for i in test]

In [24]:
test

['city',
 'city_yo',
 'run_date_yo',
 'reg_intent_yo',
 'base_yo',
 '01_daily_01_High_2024-04-14_yo',
 '01_daily_02_Medium_2024-04-14_yo',
 '02_weekly_01_High_2024-04-14_yo',
 '02_weekly_02_Medium_2024-04-14_yo',
 '02_weekly_03_Low_2024-04-14_yo',
 '03_bi_weekly_01_High_2024-04-14_yo',
 '03_bi_weekly_02_Medium_2024-04-14_yo',
 '03_bi_weekly_03_Low_2024-04-14_yo',
 '04_monthly_01_High_2024-04-14_yo',
 '04_monthly_02_Medium_2024-04-14_yo',
 '04_monthly_03_Low_2024-04-14_yo',
 '05_rare_need_01_High_2024-04-14_yo',
 '05_rare_need_02_Medium_2024-04-14_yo',
 '05_rare_need_03_Low_2024-04-14_yo',
 '01_daily_01_High_2024-04-21_yo',
 '01_daily_02_Medium_2024-04-21_yo',
 '02_weekly_01_High_2024-04-21_yo',
 '02_weekly_02_Medium_2024-04-21_yo',
 '02_weekly_03_Low_2024-04-21_yo',
 '03_bi_weekly_01_High_2024-04-21_yo',
 '03_bi_weekly_02_Medium_2024-04-21_yo',
 '03_bi_weekly_03_Low_2024-04-21_yo',
 '04_monthly_01_High_2024-04-21_yo',
 '04_monthly_02_Medium_2024-04-21_yo',
 '04_monthly_03_Low_2024-04-2

In [25]:
finaldf.to_csv('rr_stability_v0_log.csv', index=False)

In [26]:
finaldf.head(2)

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14,01_daily_01_High_2024-04-21,01_daily_02_Medium_2024-04-21,02_weekly_01_High_2024-04-21,02_weekly_02_Medium_2024-04-21,02_weekly_03_Low_2024-04-21,03_bi_weekly_01_High_2024-04-21,03_bi_weekly_02_Medium_2024-04-21,03_bi_weekly_03_Low_2024-04-21,04_monthly_01_High_2024-04-21,04_monthly_02_Medium_2024-04-21,04_monthly_03_Low_2024-04-21,05_rare_need_01_High_2024-04-21,05_rare_need_02_Medium_2024-04-21,05_rare_need_03_Low_2024-04-21,01_daily_01_High_2024-04-28,01_daily_02_Medium_2024-04-28,02_weekly_01_High_2024-04-28,02_weekly_02_Medium_2024-04-28,02_weekly_03_Low_2024-04-28,03_bi_weekly_01_High_2024-04-28,03_bi_weekly_02_Medium_2024-04-28,03_bi_weekly_03_Low_2024-04-28,04_monthly_01_High_2024-04-28,04_monthly_02_Medium_2024-04-28,04_monthly_03_Low_2024-04-28,05_rare_need_01_High_2024-04-28,05_rare_need_02_Medium_2024-04-28,05_rare_need_03_Low_2024-04-28,01_daily_01_High_2024-05-05,01_daily_02_Medium_2024-05-05,02_weekly_01_High_2024-05-05,02_weekly_02_Medium_2024-05-05,02_weekly_03_Low_2024-05-05,03_bi_weekly_01_High_2024-05-05,03_bi_weekly_02_Medium_2024-05-05,03_bi_weekly_03_Low_2024-05-05,04_monthly_01_High_2024-05-05,04_monthly_02_Medium_2024-05-05,04_monthly_03_Low_2024-05-05,05_rare_need_01_High_2024-05-05,05_rare_need_02_Medium_2024-05-05,05_rare_need_03_Low_2024-05-05
0,Chennai,2024-04-07,01_daily_01_High,33958,31422.0,1639.0,162.0,223.0,0.0,229.0,25.0,0.0,0.0,0.0,0.0,251.0,7.0,0.0,30590.0,2037.0,217.0,396.0,0.0,338.0,61.0,0.0,0.0,0.0,0.0,285.0,34.0,0.0,29559.0,2471.0,230.0,656.0,0.0,539.0,139.0,0.0,0.0,0.0,0.0,341.0,23.0,0.0,28208.0,2925.0,233.0,1006.0,2.0,944.0,255.0,6.0,12.0,7.0,1.0,318.0,41.0,0.0
1,Chennai,2024-04-07,01_daily_02_Medium,23566,1449.0,19229.0,1.0,2385.0,0.0,10.0,336.0,0.0,0.0,0.0,0.0,8.0,148.0,0.0,2415.0,17295.0,8.0,3009.0,3.0,29.0,536.0,0.0,0.0,0.0,0.0,15.0,256.0,0.0,3347.0,15438.0,13.0,3618.0,13.0,68.0,794.0,1.0,0.0,0.0,0.0,23.0,251.0,0.0,3849.0,13730.0,12.0,4284.0,33.0,124.0,1237.0,14.0,5.0,21.0,2.0,26.0,229.0,0.0


In [27]:
finaldf.to_clipboard(index=False)

## Notes

### Explanation of correction_factor Function

The correction_factor function takes two parameters: rr_weight and rapido_age.

Parameters:

rr_weight: A series or array of weights (assumed to be a pandas Series based on the usage).
rapido_age: An integer representing the age in days.

Logic and Calculations:
The function performs different calculations based on the value of rapido_age.

    Condition 1: rapido_age >= 91:
        week_inactive is set to 0.
        multiplier is set to 1.
        This means there is no inactivity, and the multiplier is 1 (no correction).
    
    Condition 2: rapido_age == 0:
        week_inactive is calculated as (91 - rapido_age) // 7 - 1.
        multiplier is calculated using the formula:
            multiplier = rr_weight.sum() / (rr_weight.sum() - (week_inactive * (week_inactive + 1) / 2))
        This corrects for the inactivity based on the number of weeks inactive.
    
    Condition 3: rapido_age % 7 == 0:
        week_inactive is calculated similarly to Condition 2.
        multiplier is calculated using the same formula as in Condition 2.
    
    Condition 4: All Other Cases:
        week_inactive is calculated as (91 - rapido_age) // 7.
        multiplier is calculated using the same formula as in Conditions 2 and 3.
        
    Return Values:
        The function returns a tuple containing:
            multiplier: The correction factor.
            week_inactive: Number of inactive weeks.
            Corrected sum of weights: rr_weight.sum() - (week_inactive * (week_inactive + 1) / 2).
            Corrected sum of squared weights: (rr_weight * rr_weight).sum() - (week_inactive * (week_inactive + 1) * (2 * week_inactive + 1) / 6).

                
Applying correction_factor to fe_data
    The line fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight, x)[0]) adds a new column rr_intent_multiplier to the DataFrame fe_data.

- For each value in the column rapido_age of fe_data, the correction_factor function is called.
- The first element of the returned tuple (i.e., multiplier) is extracted and assigned to the new column rr_intent_multiplier.

In [3]:
rapido_age = 0
rr_weight = np.log(np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]))

rr_weight_sum = rr_weight.sum()
week_inactive = (91 - rapido_age) // 7 - 1
correction = (week_inactive * (week_inactive + 1)) // 2
adjusted_sum = rr_weight_sum - correction
multiplier = rr_weight_sum / adjusted_sum

print("rr_weight_sum:", rr_weight_sum)
print("week_inactive:", week_inactive)
print("correction:", correction)
print("adjusted_sum:", adjusted_sum)
print("multiplier:", multiplier)

rr_weight_sum: 22.552163853123425
week_inactive: 12
correction: 78
adjusted_sum: -55.447836146876575
multiplier: -0.40672757352306177


Explanation of Logic

week_inactive: Number of weeks of inactivity calculated based on rapido_age.

correction: Sum of an arithmetic series representing the cumulative impact of inactive weeks.

adjusted_sum: The total sum of rr_weight after adjusting for inactivity.

multiplier: The correction factor used to adjust the weights considering inactivity.

This correction factor ensures that the impact of inactive weeks is accounted for by adjusting the total weight proportionally.

In [4]:
import numpy as np

rr_weight = np.log(np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]))
rr_weight_sum = rr_weight.sum()

rapido_ages = list(range(0, 91, 5))  # List of rapido_age values from 0 to 90 in increments of 5

for rapido_age in rapido_ages:
    if rapido_age >= 91:
        week_inactive = 0
        correction = 0
    else:
        week_inactive = (91 - rapido_age) // 7 - 1
        correction = (week_inactive * (week_inactive + 1)) // 2

    adjusted_sum = rr_weight_sum - correction
    multiplier = rr_weight_sum / adjusted_sum

    print(f"rapido_age: {rapido_age}")
    print("week_inactive:", week_inactive)
    print("correction:", correction)
    print("adjusted_sum:", adjusted_sum)
    print("multiplier:", multiplier)
    print("-" * 40)

rapido_age: 0
week_inactive: 12
correction: 78
adjusted_sum: -55.447836146876575
multiplier: -0.40672757352306177
----------------------------------------
rapido_age: 5
week_inactive: 11
correction: 66
adjusted_sum: -43.447836146876575
multiplier: -0.5190629926168298
----------------------------------------
rapido_age: 10
week_inactive: 10
correction: 55
adjusted_sum: -32.447836146876575
multiplier: -0.6950282832741157
----------------------------------------
rapido_age: 15
week_inactive: 9
correction: 45
adjusted_sum: -22.447836146876575
multiplier: -1.0046475618212922
----------------------------------------
rapido_age: 20
week_inactive: 9
correction: 45
adjusted_sum: -22.447836146876575
multiplier: -1.0046475618212922
----------------------------------------
rapido_age: 25
week_inactive: 8
correction: 36
adjusted_sum: -13.447836146876575
multiplier: -1.6770106065250832
----------------------------------------
rapido_age: 30
week_inactive: 7
correction: 28
adjusted_sum: -5.4478361468

Interpretation

rapido_age = 0: The maximum number of inactive weeks (12 weeks) results in a correction of 78, leading to a high multiplier of 7.0.

rapido_age = 5: The number of inactive weeks decreases to 11, with a correction of 66, resulting in a multiplier of approximately 3.64.

rapido_age = 10 to 90: As rapido_age increases, the number of inactive weeks and the corresponding correction decrease, leading to lower multipliers.

For rapido_age values from 80 to 90, the multiplier becomes 1.0, indicating no correction is needed since there are no inactive weeks.

## rr_regularity_final

In [28]:
#cons_type: 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
finaldf_v2 = stab_func_2(concatenated_df, base_week, base_df, 'rr_regularity_final')

In [29]:
finaldf_v2

,city,run_date,rr_regularity_final,base,01_daily_2024-04-14,02_weekly_2024-04-14,03_bi_weekly_2024-04-14,04_monthly_2024-04-14,05_rare_need_2024-04-14,01_daily_2024-04-21,02_weekly_2024-04-21,03_bi_weekly_2024-04-21,04_monthly_2024-04-21,05_rare_need_2024-04-21,01_daily_2024-04-28,02_weekly_2024-04-28,03_bi_weekly_2024-04-28,04_monthly_2024-04-28,05_rare_need_2024-04-28,01_daily_2024-05-05,02_weekly_2024-05-05,03_bi_weekly_2024-05-05,04_monthly_2024-05-05,05_rare_need_2024-05-05
0,Chennai,2024-04-07,01_daily,57524,53739.0,2771.0,600.0,0.0,414.0,52337.0,3633.0,964.0,0.0,590.0,50815.0,4530.0,1541.0,0.0,638.0,48712,5570,2580,48,614
1,Chennai,2024-04-07,02_weekly,117696,2948.0,103733.0,9974.0,0.0,1041.0,4733.0,97866.0,13571.0,0.0,1526.0,6440.0,91942.0,17893.0,0.0,1421.0,7579,85823,22503,274,1517
2,Chennai,2024-04-07,03_bi_weekly,142407,2066.0,15607.0,112798.0,10580.0,1356.0,3676.0,22290.0,100717.0,14224.0,1500.0,4821.0,28226.0,89831.0,17970.0,1559.0,5373,32011,80842,21946,2235
3,Chennai,2024-04-07,04_monthly,80231,0.0,0.0,21317.0,53805.0,5109.0,0.0,0.0,24959.0,48261.0,7011.0,476.0,731.0,28639.0,41821.0,8564.0,959,2098,29196,36532,11446
4,Chennai,2024-04-07,05_rare_need,46168,648.0,1349.0,1582.0,14710.0,25880.0,654.0,1359.0,5464.0,13735.0,21158.0,673.0,1368.0,5933.0,15621.0,18622.0,961,1525,6316,16366,16942


In [30]:
finaldf_v2.to_csv('rr_stability_v0_log_agg.csv', index=False)

In [31]:
finaldf_v2.to_clipboard(index=False)